# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
import sys
!{sys.executable} -m
%pip install -qU "langchain>=1.0" "langchain-core>=0.3" "langchain-community>=0.4" "langchain-classic>=0.3" langchain-openai langchain-elasticsearch tiktoken jq lark elasticsearch

Argument expected for the -m option
usage: /Library/Frameworks/Python.framework/Versions/3.11/Resources/Python.app/Contents/MacOS/Python [option] ... [-c cmd | -m mod | file | -] [arg] ...
Try `python -h' for more information.

[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass
from langchain_openai.embeddings import OpenAIEmbeddings
#from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever  # ← CORRECT import for 1.0+


from langchain_community.vectorstores.elasticsearch import ElasticsearchStore
#from langchain_openai import OpenAIEmbeddings

# os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [3]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")


# Create OpenAI embedding model
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Set a meaningful index name
INDEX_NAME = "workplace-knowledge-base"

# Create Elasticsearch vector store
vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings
)


/var/folders/rn/hjmgpq6j4bl5hk8kkpw6k1f40000gn/T/ipykernel_20664/3508653772.py:18: LangChainPendingDeprecationWarning: The class `ElasticsearchStore` will be deprecated in a future version. Use `Use class in langchain-elasticsearch package` instead.
  vectorstore = ElasticsearchStore(


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [4]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [5]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import json  # Optional: for metadata extraction
from datetime import datetime  # Import datetime here


def metadata_func(record: dict, metadata: dict) -> dict:
    # Populate the metadata dictionary with keys name, summary, url, category, and updated_at.
    metadata["name"]       = record.get("name", "")
    metadata["summary"]    = record.get("summary", "")
    metadata["url"]        = record.get("url", "")
    metadata["category"]   = record.get("category", "")
    metadata["updated_at"] = record.get("updated_at", "")
    return metadata


loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",  # Extracts array of records
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,       # e.g., ~750 words
    chunk_overlap=200,     # Overlap for context preservation
)

docs = loader.load_and_split(text_splitter=text_splitter)
print(f"Loaded {len(docs)} chunks")
print("Sample metadata:", docs[0].metadata)


Loaded 15 chunks
Sample metadata: {'source': '/Users/sazao/lab-chatbot-with-multi-query-retriever/temp.json', 'seq_num': 1, 'name': 'Work From Home Policy', 'summary': 'This policy outlines the guidelines for full-time remote work, including eligibility, equipment and resources, workspace requirements, communication expectations, performance expectations, time tracking and overtime, confidentiality and data security, health and well-being, and policy reviews and updates. Employees are encouraged to direct any questions or concerns', 'url': './sharepoint/Work from home policy.txt', 'category': 'teams', 'updated_at': '2020-03-01'}


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [6]:
from datetime import datetime
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

# Clean docs metadata
clean_docs = []
for doc in docs:
    metadata = doc.metadata.copy()

    if metadata.get("updated_at") in ["", None, "null"]:
        metadata["updated_at"] = datetime.now().isoformat()

    # Use model_copy() instead of copy() to avoid Pydantic deprecation warning
    clean_docs.append(doc.model_copy(update={"metadata": metadata}))

# Create embeddings ONCE
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Create vectorstore ONCE using cleaned docs
vectorstore = ElasticsearchStore.from_documents(
    clean_docs,
    embeddings,
    index_name=INDEX_NAME,
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

# Create LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=OPENAI_API_KEY # Explicitly pass the API key
)

# Create MultiQueryRetriever
retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm
)


# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [7]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain as lc_chain
import logging

# Enable detailed logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Multi-query generator with 3 variants
MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Generate 3 diverse versions of this question for better retrieval. Vary phrasing, keywords, and perspectives:

{question}

Queries (one per line):
""")

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template("""
📄 [{source}]
{page_content}
---
""")

# Define the context prompt for the LLM
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def safe_combine_docs(docs):
    """Production-ready doc formatting with fallbacks"""
    doc_strings = []
    for i, doc in enumerate(docs):
        try:
            doc_dict = doc.model_dump()
            source = doc.metadata.get("name") or doc.metadata.get("source", f"Doc-{i}")
            doc_dict["source"] = source
            formatted = LLM_DOCUMENT_PROMPT.format(**doc_dict)
        except Exception as e:
            logger.warning(f"Doc format error: {e}")
            formatted = f"[Doc-{i}] {doc.page_content[:500]}..."
        doc_strings.append(formatted)
    return "\n\n".join(doc_strings)

# Self-healing chain: retry bad retrievals
def self_healing_retriever(query, max_tries=2):
    """Retry with rewritten query if empty results"""
    for attempt in range(max_tries):
        docs = retriever.invoke(query)
        if docs:
            return docs
        logger.info(f"Empty results (attempt {attempt+1}), rewriting...")
        query = llm.invoke(f"Rewrite for better retrieval: {query}").content
    return retriever.invoke(query)  # Fallback

_context = RunnableParallel(
    context=(RunnablePassthrough() | self_healing_retriever | safe_combine_docs),
    question=RunnablePassthrough(),
)

rag_chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

# Test with auto-multi-query
def multi_query_rag(question):
    """Generate + retrieve + answer"""

    query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()

    generated_queries = query_chain.invoke({"question": question})

    print("\nGenerated Queries:")
    print("------------------")
    print(generated_queries)
    print("------------------\n")

    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]

    all_docs = []

    for q in queries:
        docs = self_healing_retriever(q)
        all_docs.extend(docs[:3])

    return rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs)
    })

print("---- Answer ----")

print(multi_query_rag("what is the nasa sales team?"))

---- Answer ----


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Generated Queries:
------------------
1. Can you provide information about the sales team at NASA?
2. How does NASA handle sales and marketing efforts?
3. What role does the sales team play within NASA's organization?
------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What details can you share about the sales department within NASA?', '2. Could you give me insights into the sales team operating within NASA?', "3. I'm interested in learning more about the sales team specifically within NASA. Can you provide relevant information?"]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration:0.115s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration

The NASA sales team consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

## Iteration 1 — Exploring more questions

Below we send a batch of different workplace-style questions through the same
`multi_query_rag` chain. Each call internally generates 3 reformulated queries,
retrieves passages for each, and asks the LLM to answer using the combined
context. Notice how questions that mix several concepts ("travel" + "policy",
"benefits" + "leave") tend to benefit the most from the multi-query expansion,
because the LLM rephrases them into more specific sub-questions.

In [8]:
questions = [
    "What is the company\'s policy on remote work?",
    "How does the paid time off policy work for new employees?",
    "What benefits are offered to employees in North America?",
    "Who is responsible for the sales team in South America?",
    "What is the dress code for client meetings?",
]

for q in questions:
    print("=" * 80)
    print("QUESTION:", q)
    print("-" * 80)
    answer = multi_query_rag(q)
    print("ANSWER:", answer)
    print()


QUESTION: What is the company's policy on remote work?
--------------------------------------------------------------------------------


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Generated Queries:
------------------
1. How does the company handle remote work arrangements?
2. Can employees work remotely at this company?
3. What are the guidelines for working remotely at this organization?
------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What policies does the company have in place for managing remote work arrangements?', "2. Can you provide insights into the company's approach to remote work arrangements?", '3. How does the company support employees working remotely?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration:0.059s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration:0.058s]
INFO:httpx:HTTP Reques

ANSWER: The company's policy on remote work allows eligible employees to work from home full time while maintaining the same level of performance and collaboration as they would in the office. Employees must have approval from their direct supervisor and the HR department to be eligible for this work-from-home arrangement.

QUESTION: How does the paid time off policy work for new employees?
--------------------------------------------------------------------------------


...

ANSWER: Gary Johnson is responsible for the sales team in South America.

QUESTION: What is the dress code for client meetings?
--------------------------------------------------------------------------------


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Generated Queries:
------------------
1. What attire is appropriate for client meetings?
2. How should I dress for client meetings?
3. Can you provide guidelines on the dress code for client meetings?
------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What is the recommended dress code for client meetings?', '2. How should one dress for professional client interactions?', '3. What clothing is considered suitable for meetings with clients?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration:0.060s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration:0.062s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings

ANSWER: The dress code for client meetings is not specified in the provided context.



## Iteration 2 — Single-query vs Multi-query retrieval

The whole point of `MultiQueryRetriever` is that a single user phrasing often
misses relevant passages because the vector store was indexed with *different*
phrasings. By generating several rewrites of the same question and unioning the
results, we widen the recall.

The cell below runs the same question through two retrievers — a vanilla
`vectorstore.as_retriever` and the `MultiQueryRetriever` we built earlier — and
prints the source titles each one surfaced. You should see the multi-query
version pulling in passages the single-query version missed.

In [9]:
single_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

question = "How is the performance of employees evaluated?"

print("--- Single-query retriever ---")
for d in single_retriever.invoke(question):
    print(" *", d.metadata.get("name", "?"), "|", d.metadata.get("category", "?"))

print()
print("--- Multi-query retriever ---")
for d in retriever.invoke(question):
    print(" *", d.metadata.get("name", "?"), "|", d.metadata.get("category", "?"))


--- Single-query retriever ---


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration:0.064s]


 * Performance Management Policy | sharepoint
 * Wfh Policy Update May 2023 | teams
 * April Work From Home Update | teams
 * Compensation Framework For It Teams | sharepoint

--- Multi-query retriever ---


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What methods are used to assess the performance of staff members?', '2. How do organizations measure the effectiveness of their employees?', '3. In what ways are employee performances reviewed and analyzed within companies?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration:0.166s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration:0.063s]
INFO:httpx:HTTP Request: POST htt

 * Performance Management Policy | sharepoint
 * Compensation Framework For It Teams | sharepoint
 * New Employee Onboarding Guide | github
 * Wfh Policy Update May 2023 | teams
 * April Work From Home Update | teams


## Iteration 3 (bonus) — Custom multi-query prompt

`MultiQueryRetriever.from_llm` can take a custom prompt. Here we instruct the
LLM to rewrite each question from three explicit perspectives (employee, HR,
manager) so that retrieval covers wording from different roles in the
workplace.

In [10]:
from langchain_core.prompts import PromptTemplate

ROLE_AWARE_PROMPT = PromptTemplate.from_template(
    """You are an AI assistant helping a search engine find relevant documents.

Rewrite the user question below into THREE different versions, each from a
distinct perspective:
  1. From the point of view of an EMPLOYEE asking about their own situation.
  2. From the point of view of HR documenting the official policy.
  3. From the point of view of a MANAGER applying the rule to their team.

Original question: {question}

Output (one rewrite per line, no numbering, no extra commentary):"""
)

role_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm,
    prompt=ROLE_AWARE_PROMPT,
)

question = "What happens if I take more vacation days than I have available?"
docs = role_retriever.invoke(question)

print("Retrieved", len(docs), "passages with the role-aware retriever:")
for d in docs:
    print(" *", d.metadata.get("name", "?"), "|", d.metadata.get("category", "?"))


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What are the consequences if I exceed my allocated vacation days?', '2. What is the company policy regarding exceeding the allotted vacation days?', '3. How should I handle a situation where an employee takes more vacation days than they have available?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration:0.055s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://fdbe58bc6b5249648233e31cc268477f.europe-west1.gcp.cloud.es.io:443/workplace-knowledge-base/_search?_source_includes=metadata,text [status:200 duration:0.057s]
INF

Retrieved 4 passages with the role-aware retriever:
 * Company Vacation Policy | sharepoint
 * April Work From Home Update | teams
 * Wfh Policy Update May 2023 | teams
 * Work From Home Policy | teams
